# 11 Deep Learning (PyTorch): Predicting Infection with a Neural Network

A colleague asks: "Should we try deep learning?"
We'll build a simple binary classification network with PyTorch and see how far we can get on 280 rows of data.

Workflow: **data preprocessing → model architecture → training loop → early stopping → AUC evaluation → learning curve → comparison with sklearn**

In [ ]:
# Google Colab setup -- skip this cell if running locally
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

## Step 1 — Data preprocessing: turning the patient table into tensors PyTorch understands

PyTorch doesn't eat DataFrames -- it only eats **tensors**. This step converts categorical columns to numbers, standardizes numeric columns, makes sure the dtype is `float32`, and finally hand-splits a training set and a validation set -- things sklearn usually handles in one line of `Pipeline`. Here we deliberately pull it apart and do it by hand, so you can see exactly what each step does.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)` | Creates the binary outcome variable (0/1); like Ch10, symptoms are excluded as features to avoid data leakage |
> | `X_df = pd.get_dummies(df[...], drop_first=True)` | One-hot encodes categorical columns (`sex`, `wing`...) into 0/1 dummy variables; `drop_first=True` avoids collinearity |
> | `X_np = X_df.values.astype(np.float32)` | Converts to a NumPy array and casts to **`float32`** -- PyTorch's weights default to single precision, so the dtype must match or you'll get a dtype error |
> | `scaler.fit_transform(X_np[:, 0:1])` | Standardizes only `age` (mean 0, std 1); neural networks are sensitive to input scale, and skipping standardization can slow or even break convergence |
> | `np.random.shuffle(idx)` → `train_idx, val_idx = idx[:split], idx[split:]` | Manually shuffles the indices and splits 70/30 into training / validation -- `sklearn.train_test_split` isn't used here on purpose, to demonstrate the mechanics of splitting |
> | `y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)` | `unsqueeze(1)` reshapes from `(N,)` to `(N, 1)`, matching the model's output shape -- otherwise the loss function would silently broadcast and compute the wrong loss |

> ⚠️ **Hidden leakage in small data**: here `StandardScaler` is `fit` on **all 280 rows** before the train/validation split -- strictly speaking, the validation set's statistics have quietly "leaked" into the standardization parameters, a very mild form of data leakage. With 280 rows and only one numeric column the leak is small enough to ignore, and the point here is purely to demonstrate the manual workflow; for a real project, follow Ch10 and put preprocessing inside a `Pipeline`, `fit` only on the training fold. Also, because the split is random, the `torch.manual_seed(42)` / `np.random.seed(42)` at the top ensures every rerun gets the same split and the same initial weights, so results are reproducible and can be fairly compared against sklearn.

In [ ]:
# --- Step 1: Data preprocessing (converting to tensors by hand) ---
import pathlib

import pandas as pd
import numpy as np
import torch
from torch import nn
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# -- CJK font setup (prevents Chinese labels from rendering as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

torch.manual_seed(42)
np.random.seed(42)

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

# Same features as Ch10 (no symptoms, to avoid data leakage)
num_cols = ["age"]
cat_cols = ["sex", "smoking_history", "functional_status", "wing"]
bin_cols = [
    "floor", "comorbidity_chf", "comorbidity_dm", "comorbidity_cancer",
    "comorbidity_copd", "immunosuppressed", "shower_use", "hydrotherapy_use",
]

# One-hot encode categorical features
X_df = pd.get_dummies(df[num_cols + cat_cols + bin_cols], drop_first=True)
# Convert to a NumPy array and cast to float32 (PyTorch weights default to single precision, so dtypes must match)
X_np = X_df.values.astype(np.float32)
y_np = df["infected"].values.astype(np.float32)

# Standardize age
scaler = StandardScaler()
X_np[:, 0] = scaler.fit_transform(X_np[:, 0:1]).ravel()

# 70/30 split
idx = np.arange(len(X_np))
np.random.shuffle(idx)
split = int(0.7 * len(idx))
train_idx, val_idx = idx[:split], idx[split:]

# unsqueeze(1) turns y from 1-D (N,) into 2-D (N, 1), matching the model's output shape
X_train = torch.tensor(X_np[train_idx])
y_train = torch.tensor(y_np[train_idx]).unsqueeze(1)
X_val = torch.tensor(X_np[val_idx])
y_val = torch.tensor(y_np[val_idx]).unsqueeze(1)

print(f"Feature dimensions: {X_train.shape[1]}")
print(f"Training set: {len(X_train)}, validation set: {len(X_val)}")
print(f"Feature names: {list(X_df.columns)}")

## Step 2 — Model architecture: stacking three linear layers into a minimal neural network

`nn.Sequential` is PyTorch's simplest container -- it chains layers one after another, and data flows through them from top to bottom. Here we stack two rounds of "linear transform + ReLU nonlinearity," and the last layer outputs a single number (a logit) directly, with no sigmoid attached.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `nn.Sequential(nn.Linear(input_dim, 32), nn.ReLU(), nn.Linear(32, 16), nn.ReLU(), nn.Linear(16, 1))` | Stacks the layers into a pipeline: `Linear` does a linear combination, `ReLU` adds nonlinearity, and the final `Linear(16, 1)` outputs a single value |
> | `nn.ReLU()` | The nonlinear activation -- without it, stacking any number of `Linear` layers is mathematically equivalent to a single linear regression |
> | `n_params = sum(p.numel() for p in model.parameters())` | Sums the element count of all weights + biases, quantifying "how many knobs this model has to tune" |

> 💡 **Why doesn't the last layer have a sigmoid?** Because Step 3 uses `nn.BCEWithLogitsLoss`, which folds sigmoid and binary cross-entropy together internally, and is numerically more stable than "manually applying sigmoid, then computing BCE" (it avoids extreme values like `log(0)` blowing up). So the model's only job is to output a raw logit; sigmoid only shows up when we compute probabilities (Step 5) or the loss (Step 3).

In [ ]:
# --- Step 2: Model architecture ---
# input_dim → 32 → 16 → 1
input_dim = X_train.shape[1]

# Just stack Linear + ReLU; the last layer has no activation (it outputs a raw logit, left for BCEWithLogitsLoss to handle)
model = nn.Sequential(
    nn.Linear(input_dim, 32),
    nn.ReLU(),
    nn.Linear(32, 16),
    nn.ReLU(),
    nn.Linear(16, 1),
)

# Number of parameters
n_params = sum(p.numel() for p in model.parameters())
print(f"Model architecture: {input_dim} → 32 → 16 → 1")
print(f"Total parameters: {n_params}")
print(f"Parameter / sample ratio: {n_params / len(X_train):.1f}")
print(f"\n→ More parameters than samples → very high overfitting risk!")

## Step 3 — Training loop + early stopping: 3 verbs run up to 300 times, but we quit while we're ahead

A PyTorch training loop always keeps the same rhythm: **forward** (compute predictions) → **loss** (compute error) → **backward** (compute gradients) → **step** (update parameters). Here we add early stopping on top: monitor the validation loss while training, and as soon as `patience` rounds pass without improvement, stop early and roll back to the best-performing set of weights.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `optimizer.zero_grad()` | Core verb 1 of 3: clears gradients left over from the previous round (PyTorch accumulates gradients by default, and not clearing them makes them keep growing) |
> | `logits = model(X_train)` | forward: data flows through the network, producing raw logits (not yet passed through sigmoid) |
> | `loss = loss_fn(logits, y_train)` | Uses `BCEWithLogitsLoss` to compare predictions against true labels and compute this round's error |
> | `loss.backward()` | Core verb 2 of 3: backprop -- PyTorch's autodiff computes the gradient of the loss with respect to every parameter |
> | `optimizer.step()` | Core verb 3 of 3: updates every parameter using the gradient just computed (following the Adam optimizer's rule) |
> | `if val_loss < best_val_loss: ... counter = 0 else: counter += 1` | The core of early stopping: save a checkpoint and reset `counter` to zero when val_loss improves; otherwise increment `counter` |
> | `model.load_state_dict(best_state)` | After training ends, "rewind" to the weight snapshot with the lowest val_loss, instead of using the weights from the final round (which may already be overfit) |

> 🧭 **Quit while you're ahead**: `patience` is "how many more rounds to give it," and `best_state` is "the save point for the current best performance." Once `patience` rounds pass in a row without setting a new record, training stops and the trophy goes to the best-performing round -- not to whatever the model looks like the moment training ends (which may already have started overfitting).

In [ ]:
# --- Step 3: Training loop + early stopping ---
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Record history
train_losses, val_losses = [], []
best_val_loss = float("inf")
patience, counter = 15, 0
best_state = None
best_epoch = 0

for epoch in range(300):
    # Train
    model.train()
    # Core verb 1 of 3: clear gradients left over from the previous round
    optimizer.zero_grad()
    # forward: data flows through the network, producing raw logits
    logits = model(X_train)
    loss = loss_fn(logits, y_train)
    # Core verb 2 of 3: backprop -- PyTorch's autodiff computes the gradient for every parameter
    loss.backward()
    # Core verb 3 of 3: update each parameter using the gradient just computed
    optimizer.step()
    train_losses.append(loss.item())

    # Validate
    model.eval()
    with torch.no_grad():
        val_logits = model(X_val)
        val_loss = loss_fn(val_logits, y_val).item()
    val_losses.append(val_loss)

    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        # Save a snapshot of the current best weights (clone so it isn't overwritten later)
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        counter = 0
    else:
        counter += 1

    if counter >= patience:
        print(f"Early stopping at epoch {epoch}")
        break

# Load the best model
model.load_state_dict(best_state)
print(f"Best epoch: {best_epoch}, best val_loss: {best_val_loss:.4f}")

## Step 4 — Learning curve: plotting the training process

`train_losses` and `val_losses` were already recorded round by round inside the Step 3 loop; here we just plot them as two lines, making it easy to see at a glance how the gap between the training and validation sets changes across epochs.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `ax.plot(train_losses, ...)` / `ax.plot(val_losses, ...)` | Plots the loss recorded at each epoch as two lines, comparing the training / validation trends |
> | `ax.axvline(x=best_epoch, ...)` | Marks the best epoch chosen by early stopping, making it easy to see what was happening on the curve at that point |

> 💡 **Reading the curve for overfitting**: if the train loss keeps sliding down but the val loss first drops and then rises -- a "V-shaped rebound" -- that rebound point is the signal that the model has started memorizing the training data. What early stopping does is stop right around that rebound point.

In [ ]:
# --- Step 4: Learning curve ---
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#2c7fb8")
ax.plot(val_losses, label="Val Loss", color="#e34a33")
# Mark the best epoch chosen by early stopping
ax.axvline(x=best_epoch, color="gray", linestyle="--", alpha=0.5,
           label=f"Best epoch ({best_epoch})")
ax.set_xlabel("Epoch")
ax.set_ylabel("BCEWithLogitsLoss")
ax.set_title("Learning Curve")
ax.legend()
plt.tight_layout()
plt.show()

print("→ If train loss keeps falling but val loss rebounds → overfitting")
print("→ Early stopping halts training when val loss stops improving")

## Step 5 — Evaluation: computing AUC on the validation set

During training the model is fed raw logits, but AUC needs "probability-like scores," so at evaluation time we manually apply `sigmoid` once more, converting logits back into probabilities between 0 and 1, and then score them with the same `roc_auc_score` used in Ch10.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `with torch.no_grad():` | Evaluation doesn't need gradients, so turning off gradient tracking saves memory and speeds up computation |
> | `torch.sigmoid(model(X_val)).numpy()` | Passes the model's output logits through sigmoid to get probabilities, so they can be fed into `roc_auc_score` |
> | `roc_auc_score(y_val.numpy(), val_proba)` | Compares the "probability ranking" against the "true labels" to compute AUC -- the same metric used in Ch10, so the comparison is fair |
> | `auc_train - auc_val` | The gap between training-set and validation-set AUC is the most direct overfitting detector |

> ⚠️ **Train with logits, evaluate with probabilities**: `BCEWithLogitsLoss` already applies sigmoid internally, so feeding it raw logits directly during training is more efficient and numerically stable; but computing AUC needs human-readable probability scores, which is why `torch.sigmoid()` is applied manually here.

In [ ]:
# --- Step 5: AUC evaluation ---
model.eval()
# No need to compute gradients during evaluation; turning off gradient tracking saves memory and speeds things up
with torch.no_grad():
    # Pass logits through sigmoid to get probabilities in [0, 1], so they can be fed into roc_auc_score
    val_proba = torch.sigmoid(model(X_val)).numpy()
    train_proba = torch.sigmoid(model(X_train)).numpy()

auc_train = roc_auc_score(y_train.numpy(), train_proba)
auc_val = roc_auc_score(y_val.numpy(), val_proba)

print(f"=== PyTorch DL results ===")
print(f"Train AUC = {auc_train:.3f}")
print(f"Val   AUC = {auc_val:.3f}")
print(f"Gap       = {auc_train - auc_val:.3f}")

if auc_train - auc_val > 0.1:
    print("\n→ Train-Val gap > 0.1 → severe overfitting")
    print("→ 280 rows aren't enough to support this model's parameter count")
else:
    print("\n→ Gap is small; the model is relatively stable")

## Step 6 — Comparison with sklearn: is DL really stronger?

Looking at DL's own AUC in isolation is meaningless -- it has to be compared against Ch10's Logistic Regression and Random Forest under **exactly the same** train/val split, to know whether stacking a few extra neural network layers was even worth it.

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `X_full = df[num_cols + cat_cols + bin_cols]` | Feeds the raw (not one-hot-encoded, not standardized) columns to sklearn, leaving preprocessing to a unified `ColumnTransformer` |
> | `X_sk_train = X_full.iloc[train_idx]` | Reuses the **exact same** `train_idx` / `val_idx` split from Step 1, ensuring all three models are compared on the same exam |
> | `Pipeline([("pre", preprocess), ("model", LogisticRegression(...))])` | Same as Ch10: wraps preprocessing and the model into a single Pipeline |
> | `compare.append(("PyTorch DL", auc_val))` | Folds the DL validation AUC computed in Step 5 into the same comparison table |

> 🧭 **The key to a fair comparison is "the same split"**: if the three models each used a different train/val split, differences in AUC could just be luck (which rows happened to land in the validation set) rather than a real difference between the models. That's why this notebook stubbornly keeps reusing `train_idx` / `val_idx`.

In [ ]:
# --- Step 6: Comparison with sklearn ---
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Use the same train/val split
X_full = df[num_cols + cat_cols + bin_cols]
y_full = df["infected"]

# Reuse the same train_idx / val_idx from Step 1, so all three models are compared on the same exam
X_sk_train = X_full.iloc[train_idx]
X_sk_val = X_full.iloc[val_idx]
y_sk_train = y_full.iloc[train_idx]
y_sk_val = y_full.iloc[val_idx]

# Same as Ch10: standardize numeric features + one-hot encode categoricals + pass binary columns through unchanged
preprocess = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore", drop="first"), cat_cols),
    ("bin", "passthrough", bin_cols),
])

compare = []

# Logistic Regression
clf_lr = Pipeline([("pre", preprocess), ("model", LogisticRegression(max_iter=500, random_state=42))])
clf_lr.fit(X_sk_train, y_sk_train)
auc_lr = roc_auc_score(y_sk_val, clf_lr.predict_proba(X_sk_val)[:, 1])
compare.append(("Logistic Regression", auc_lr))

# Random Forest
clf_rf = Pipeline([("pre", preprocess), ("model", RandomForestClassifier(n_estimators=100, random_state=42))])
clf_rf.fit(X_sk_train, y_sk_train)
auc_rf = roc_auc_score(y_sk_val, clf_rf.predict_proba(X_sk_val)[:, 1])
compare.append(("Random Forest", auc_rf))

# PyTorch
compare.append(("PyTorch DL", auc_val))

print("=== Model comparison (same train/val split) ===")
for name, auc in compare:
    print(f"  {name:25s}  Val AUC = {auc:.3f}")

print("\n→ On 280 rows, the three models usually perform very similarly")
print("→ DL shows no clear advantage and instead carries overfitting risk")
print("→ Educational value: learn PyTorch syntax so it pays off later on large datasets")

## Bonus: model interpretation — which clues is this neural network actually using?

A DL model is a black box -- there are no coefficients to read directly the way there are with logistic regression. Here we use **hand-rolled permutation importance**: shuffle the values in one column of the validation set, recompute AUC, and the bigger the drop, the more important that column is to the model. This is the same concept as `sklearn.inspection.permutation_importance` from Ch10, implemented by hand here (and deliberately without SHAP-on-torch -- that runs slowly and error-prone in a headless CI environment, whereas the manual shuffling approach finishes in seconds, is stable, and needs no extra dependencies).

> **Line-by-line**:
>
> | This line | What it does |
> |---|---|
> | `baseline_auc = roc_auc_score(y_val.numpy(), baseline_proba)` | First computes the original AUC with nothing shuffled, as the point of comparison |
> | `for col_idx, col_name in enumerate(X_df.columns):` | Shuffles each column in turn, touching only one column at a time while leaving the rest unchanged |
> | `rng.shuffle(X_shuffled[:, col_idx])` | Shuffles that column's values across rows, equivalent to showing the model data where "this column's information has been erased, everything else unchanged" |
> | `shuffled_auc = roc_auc_score(y_val.numpy(), shuffled_proba)` | Recomputes AUC on the shuffled data |
> | `drops.append(baseline_auc - shuffled_auc)` | The bigger the AUC drop, the more important that column is to the model's predictions; repeating `n_repeats` times and averaging avoids one shuffle's luck skewing the result |

> 🧭 **Important ≠ causal**: permutation importance tells you "how much the model relies on this clue to predict," not "changing this feature would change infection risk." To actually ask about causation (e.g., whether shower exposure "causes" infection), look at the confounder-adjusted OR from Ch06, or the causal inference methods in Ch12 -- a feature with strong predictive power might just be a bystander that happens to co-occur with the real cause.

In [ ]:
# --- Bonus: hand-computing Permutation Importance ---
# Idea: shuffle the values in one column -- if that column was really important, the model will "lose its memory" and AUC drops noticeably;
# if that column was useless, AUC barely changes before and after shuffling.

rng = np.random.default_rng(42)
n_repeats = 10

model.eval()
with torch.no_grad():
    baseline_proba = torch.sigmoid(model(X_val)).numpy()
baseline_auc = roc_auc_score(y_val.numpy(), baseline_proba)

X_val_np = X_val.numpy()  # Convert back to NumPy, so it's easy to shuffle column by column
records = []

for col_idx, col_name in enumerate(X_df.columns):
    drops = []
    for _ in range(n_repeats):
        X_shuffled = X_val_np.copy()
        rng.shuffle(X_shuffled[:, col_idx])  # Shuffle only this column; leave the others untouched
        with torch.no_grad():
            shuffled_proba = torch.sigmoid(model(torch.tensor(X_shuffled))).numpy()
        shuffled_auc = roc_auc_score(y_val.numpy(), shuffled_proba)
        drops.append(baseline_auc - shuffled_auc)
    records.append((col_name, float(np.mean(drops)), float(np.std(drops))))

imp_df = pd.DataFrame(records, columns=["feature", "importance", "std"])
imp_df = imp_df.sort_values("importance", ascending=False).reset_index(drop=True)

print(f"Baseline Val AUC = {baseline_auc:.3f}\n")
print("=== Permutation Importance (AUC drop, bigger = more important) ===")
print(imp_df.head(8).to_string(index=False))

## Summary

| Step | Skill learned |
|------|------------|
| Preprocessing | `pd.get_dummies()` + `torch.tensor()` manual conversion |
| Model | `nn.Sequential(Linear → ReLU → Linear → ReLU → Linear)` |
| Training loop | `zero_grad → forward → loss → backward → step` |
| Early stopping | Monitor val_loss; stop once patience runs out |
| Learning curve | Visualize train/val loss to diagnose overfitting |
| Model comparison | Fair comparison of DL vs sklearn on the same split |

**Bottom line**:
- 280 rows → DL is overkill; sklearn is enough
- But PyTorch syntax is worth learning: you'll need it later for images, sequences, and large samples
- The point isn't "which model is strongest" but "using the right tool for the right problem"

In the next chapter (Ch12), we ask: did showering really "cause" the infections? → Causal inference.